In [2]:
from langchain_community.document_loaders import TextLoader # takes raw text and converts into text that langchain can read
from langchain_text_splitters import CharacterTextSplitter # splits desc document into meaningful chunks
from langchain_openai import OpenAIEmbeddings # converting chunks into document embeddings for api calls
from langchain_chroma import Chroma # store in vector database

In [16]:
from dotenv import load_dotenv
load_dotenv()

True

In [17]:
# read in dataset
import pandas as pd

books = pd.read_csv("books_cleaned.csv")

In [18]:
books

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 A NOVEL THAT READERS and critics...
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine..."
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897 Lewis' work on the nature of lov...
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5192,9788172235222,8172235224,Mistaken Identity,Nayantara Sahgal,Indic fiction (English),http://books.google.com/books/content?id=q-tKP...,On A Train Journey Home To North India After L...,2003.0,2.93,324.0,0.0,Mistaken Identity,9788172235222 On A Train Journey Home To North...
5193,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,This book tells the tale of a man who goes on ...,2002.0,3.70,175.0,24.0,Journey to the East,9788173031014 This book tells the tale of a ma...
5194,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,1568.0,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623 Wisdom to Create a Life of Passi...
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,104.0,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535 This collection of the timeless ...


In [19]:
books["tagged_description"]

0       9780002005883 A NOVEL THAT READERS and critics...
1       9780002261982 A new 'Christie for Christmas' -...
2       9780006178736 A memorable, mesmerizing heroine...
3       9780006280897 Lewis' work on the nature of lov...
4       9780006280934 "In The Problem of Pain, C.S. Le...
                              ...                        
5192    9788172235222 On A Train Journey Home To North...
5193    9788173031014 This book tells the tale of a ma...
5194    9788179921623 Wisdom to Create a Life of Passi...
5195    9788185300535 This collection of the timeless ...
5196    9789027712059 Since the three volume edition o...
Name: tagged_description, Length: 5197, dtype: str

In [10]:
# put tagged description into its own csv text file
books["tagged_description"].to_csv("tagged_description.txt",
                                   index = False,
                                   header = False)

In [39]:
# start splitting data by loading and instantiating text split up
raw_documents = TextLoader("tagged_description.txt").load()
text_splitter = CharacterTextSplitter(chunk_size=0.1, chunk_overlap=0, separator="\n") # chunk size = 0 to prioritize splitting on seperator and not chunk size, no overlap for chunks

# done w instantiation of text splitter, now apply to documents
documents = text_splitter.split_documents(raw_documents)

Created a chunk of size 1170, which is longer than the specified 0
Created a chunk of size 1216, which is longer than the specified 0
Created a chunk of size 375, which is longer than the specified 0
Created a chunk of size 311, which is longer than the specified 0
Created a chunk of size 483, which is longer than the specified 0
Created a chunk of size 484, which is longer than the specified 0
Created a chunk of size 962, which is longer than the specified 0
Created a chunk of size 188, which is longer than the specified 0
Created a chunk of size 845, which is longer than the specified 0
Created a chunk of size 296, which is longer than the specified 0
Created a chunk of size 197, which is longer than the specified 0
Created a chunk of size 881, which is longer than the specified 0
Created a chunk of size 1090, which is longer than the specified 0
Created a chunk of size 1191, which is longer than the specified 0
Created a chunk of size 306, which is longer than the specified 0
Create

In [40]:
documents[0]

Document(metadata={'source': 'tagged_description.txt'}, page_content='"9780002005883 A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, G

In [41]:
##### CREATE VECTOR DATABASE
db_books= Chroma.from_documents(documents, embedding=OpenAIEmbeddings())

In [54]:
##### START OF QUERYING SECTION (YAY)

query = "A gritty, morally grey world full of heroes and villains"
# get recommendations
docs = db_books.similarity_search(query, k = 10) # k is the number of results search returns
docs

[Document(id='c03980d5-a199-455e-a52d-9806d8a24b3c', metadata={'source': 'tagged_description.txt'}, page_content='9780380720866 A dread darkness is descending upon a great land called Midkemia—a powerful and malevolent race of monsters that has slipped through a hole in the dimensions. And two unlikely young heroes—a bastard heir denied his birthright and an irrepressible scoundrel with a penchant for thievery—must take up arms in the struggle to protect their besieged world...two friends chosen by Destiny to stand at the fore of the battle that is to come against the gargantuan reptile army of the terrible Emerald Queen.'),
 Document(id='aa6e425f-8065-4230-b04c-fbc7fcae5fea', metadata={'source': 'tagged_description.txt'}, page_content='9780380720866 A dread darkness is descending upon a great land called Midkemia—a powerful and malevolent race of monsters that has slipped through a hole in the dimensions. And two unlikely young heroes—a bastard heir denied his birthright and an irrepr

In [55]:
# extract title from the description using the isbn number
isbn_str = docs[0].page_content.split()[0].strip().strip('"')
isbn_num = int(isbn_str)

books[books["isbn13"] == isbn_num]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
1759,9780380720866,0380720868,Shadow of a Dark Queen,Raymond E. Feist,Fiction,http://books.google.com/books/content?id=j5Yyl...,A dread darkness is descending upon a great la...,1995.0,4.04,497.0,22935.0,Shadow of a Dark Queen: Book One of the Serpen...,9780380720866 A dread darkness is descending u...


In [64]:
# bundle into a function so it gives us all the recommendations
def get_semantic_recommendations(query: str,
        top_k: int = 10,
) -> pd.DataFrame:
    # get recommendations
    recs = db_books.similarity_search(query, k = 50)

    # create list to hold all the isbn of recommendations
    books_list = []
    for i in range(0, len(recs)):
        books_list += [int(recs[i].page_content.strip('"').split()[0])]

    # return books dataset that matches the isbn (all the recommended books)
    return books[books["isbn13"].isin(books_list)]


In [65]:
get_semantic_recommendations("A book about pirates and sea adventures")

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
19,9780006499169,0006499163,Post Captain,Patrick O'Brian,"Aubrey, Jack (Fictitious character)",http://books.google.com/books/content?id=S761k...,This tale begins with Jack Aubrey arriving hom...,1996.0,4.29,474.0,7815.0,Post Captain,9780006499169 This tale begins with Jack Aubre...
198,9780060733353,0060733357,The Confusion,Neal Stephenson,Fiction,http://books.google.com/books/content?id=NJbll...,"In the year 1689, a cabal of Barbary galley sl...",2005.0,4.25,815.0,18790.0,The Confusion: Volume Two of The Baroque Cycle,"9780060733353 In the year 1689, a cabal of Bar..."
916,9780156032599,0156032597,The Pirates Laffite,William C. Davis,Biography & Autobiography,http://books.google.com/books/content?id=Yqpwm...,From just after the Louisiana Purchase through...,2006.0,3.51,756.0,211.0,The Pirates Laffite: The Treacherous World of ...,9780156032599 From just after the Louisiana Pu...
941,9780192781536,0192781537,"Moby Dick, Or, The White Whale",Herman Melville;Geraldine McCaughrean;Victor G...,Juvenile Fiction,http://books.google.com/books/content?id=La1AS...,Retells Melville's novel of the sailor called ...,1998.0,3.85,104.0,38.0,"Moby Dick, Or, The White Whale",9780192781536 Retells Melville's novel of the ...
1074,9780237525378,0237525372,Oliver Twist,Pauline Francis;Charles Dickens,Juvenile Nonfiction,http://books.google.com/books/content?id=X6RvT...,This wonderful series is a quick way into a ra...,2003.0,3.66,48.0,92.0,Oliver Twist,9780237525378 This wonderful series is a quick...
1075,9780237526887,0237526883,"20,000 Leagues Under the Sea",Jules Verne;Pauline Francis,Children's stories,http://books.google.com/books/content?id=qnNyp...,The abridged adventures of a French professor ...,2004.0,3.76,48.0,543.0,"20,000 Leagues Under the Sea",9780237526887 The abridged adventures of a Fre...
1276,9780312864743,0312864744,Ports of Call,Jack Vance,Fiction,http://books.google.com/books/content?id=_Zu4S...,A romantic tale follows a space swashbuckler a...,1999.0,3.53,300.0,395.0,Ports of Call,9780312864743 A romantic tale follows a space ...
1746,9780375829635,0375829636,Young Warriors,Tamora Pierce;Josepha Sherman,Juvenile Fiction,http://books.google.com/books/content?id=6EnZw...,Fifteen original short stories by various auth...,2005.0,3.91,312.0,3583.0,Young Warriors: Stories of Strength,9780375829635 Fifteen original short stories b...
1913,9780393321074,039332107X,Blue at the Mizzen (Vol. Book 20) (Aubrey/Matu...,Patrick O'Brian,Fiction,http://books.google.com/books/content?id=rfSrf...,"Now that the Napoleonic Wars are over, frigate...",2000.0,4.38,262.0,2191.0,Blue at the Mizzen (Vol. Book 20) (Aubrey/Matu...,9780393321074 Now that the Napoleonic Wars are...
2039,9780425193723,0425193721,The Sea Hunters II,Clive Cussler;Craig Dirgo,History,http://books.google.com/books/content?id=shU0A...,Collects accounts of the underwater discoverie...,2003.0,3.84,464.0,1286.0,The Sea Hunters II,9780425193723 Collects accounts of the underwa...
